In [ ]:
#importing session, env-vars, functions and others

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f
import pandas as pd
from collections import defaultdict, OrderedDict

import flatdict as flatdict
import os
from dotenv import load_dotenv

env = load_dotenv('/mnt/d/linux/project/vars/.env')


In [ ]:
#loaded env_vars
pgjar = os.getenv('POSTGRE_JAR')
pgconn = os.getenv('POSTGRE_CONN')
pgdriver = os.getenv('PG_DRIVER')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')

In [ ]:
#create spark session
spark = ((
        SparkSession
            .builder
            .appName('vehicle_temp')
            .config('spark.jars', f'{pgjar}')
            .config('spark.driver.extraClassPath', pgjar)
            .config('spark.executor.extraClassPath', pgjar)
        .getOrCreate()
))



In [ ]:
#select data from table and manupulating them in dataframe

tmpvehicle_frame = (( 
                     spark
                        .read
                        .format('jdbc')
                        .option('url', pgconn)
                        .option('driver', pgdriver)
                        .option('user', pguser)
                        .option('password', pgpwd)
                        .option('query', 
                                
                                """ 
                                    select 
                                    
                                        vehicletemperatureid,
                                        temperature,
                                        fullsensordata,
                                        recordedwhen,
                                        vehicleregistration,
                                        _airbyte_ab_id,
                                        _airbyte_emitted_at,
                                        _airbyte_normalized_at,
                                        _airbyte_vehicletemperatures_hashid
                                        
                                    from warehouse.vehicletemperatures
                                
                                """
                                )
                    )).load()

tmpvehicle_frame.show(5, False)

In [ ]:

norm_jsoncol = (( 
                 tmpvehicle_frame
                 .select('fullsensordata')
                 .toPandas()
))

newcol = norm_jsoncol.to_dict()
# dff = pd.DataFrame(norm_jsoncol)

\newcol = pd.json_normalize(newcol, 'Recordings').pd.melt()

newcol

In [ ]:
for d in dff:
    df_tmp = pd.json_normalize(dff)
    row = pd.DataFrame(df_tmp.to_numpy().flatten()).T.dropna(axis=1)
    row.columns = df_tmp.columns
    df = pd.concat([df, row])

print(df.reset_index(drop=True))

In [ ]:
def flatten_json(y):
    
    out = {}

    def flatten(x, name=''):
        
        if type(x) is dict:
            for a in x:
                
                flatten(x[a], name + a + '_')
                
        elif type(x) is list:
            i = 0
            
            for a in x:
                
                flatten(a, name + str(i) + '_')
                
        else:
            out[str(name[:-1])] = str(x)
            
    flatten(y)
    
    return (out)                

In [ ]:
var = apply_normalizations(new)